# Paper Results Display

This notebook loads generated artifacts from `research/outputs/` and displays paper-ready sections:

- Dataset description
- Training details
- FFNN metrics
- Latency and resource usage
- Qualitative semantic examples and maintenance reports

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Prefer local folder auto-detection, with explicit fallback for reproducibility.
CANDIDATE_ROOTS = [
    Path.cwd(),
    Path('/Users/samarthatreya/Documents/Claped/PISA-Clap'),
]

PROJECT_ROOT = None
for candidate in CANDIDATE_ROOTS:
    if (candidate / 'research' / 'outputs').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = CANDIDATE_ROOTS[-1]

OUTPUT_DIR = PROJECT_ROOT / 'research' / 'outputs'

paths = {
    'metrics': OUTPUT_DIR / 'metrics.json',
    'summary': OUTPUT_DIR / 'paper_summary.md',
    'qualitative': OUTPUT_DIR / 'qualitative_examples.json',
    'latency_csv': OUTPUT_DIR / 'latency_metrics.csv',
    'confusion_csv': OUTPUT_DIR / 'confusion_matrix.csv',
}

def show_missing(path_key: str):
    display(Markdown(f"> ⚠ Missing artifact: `{path_key}` at `{paths[path_key]}`"))

def safe_float(value, default=np.nan):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

print('Project root:', PROJECT_ROOT)
print('Output directory:', OUTPUT_DIR)
for key, path in paths.items():
    print(f"{key:>14}: {'OK' if path.exists() else 'MISSING'}")


: 

In [ ]:
metrics = {}
if paths['metrics'].exists():
    metrics = json.loads(paths['metrics'].read_text(encoding='utf-8'))
else:
    show_missing('metrics')

summary_text = ''
if paths['summary'].exists():
    summary_text = paths['summary'].read_text(encoding='utf-8')
else:
    show_missing('summary')

if paths['latency_csv'].exists():
    latency_df = pd.read_csv(paths['latency_csv'])
else:
    latency_df = pd.DataFrame(columns=['mode', 'mean_ms', 'std_ms', 'n'])
    show_missing('latency_csv')

if paths['confusion_csv'].exists():
    confusion_df = pd.read_csv(paths['confusion_csv'], index_col=0)
else:
    confusion_df = pd.DataFrame(index=['True Normal', 'True Abnormal'], columns=['Pred Normal', 'Pred Abnormal']).fillna(0)
    show_missing('confusion_csv')

if paths['qualitative'].exists():
    qualitative_raw = json.loads(paths['qualitative'].read_text(encoding='utf-8'))
else:
    qualitative_raw = []
    show_missing('qualitative')

qualitative_df = pd.DataFrame(qualitative_raw)

dataset = metrics.get('dataset', {})
training = metrics.get('training', {})
reliability = metrics.get('reliability', {})
latency_metrics = metrics.get('latency', {})

# Prefer CSV latency for display consistency; fall back to metrics.json if CSV is empty.
if latency_df.empty and latency_metrics:
    latency_df = pd.DataFrame([
        {'mode': mode, **stats}
        for mode, stats in latency_metrics.items()
    ])

latency_df['mean_ms'] = latency_df.get('mean_ms', pd.Series(dtype=float)).map(safe_float)
latency_df['std_ms'] = latency_df.get('std_ms', pd.Series(dtype=float)).map(lambda x: safe_float(x, 0.0))
latency_df['n'] = latency_df.get('n', pd.Series(dtype=float)).map(lambda x: safe_float(x, np.nan))

resource_stats = {
    'device': 'N/A',
    'runtime_seconds': np.nan,
    'max_rss_mb': np.nan,
}
if summary_text:
    device_match = re.search(r"^- Device used by torch:\s*(.+)$", summary_text, re.MULTILINE)
    runtime_match = re.search(r"^- End-to-end evaluation runtime:\s*([0-9.]+)\s*s", summary_text, re.MULTILINE)
    rss_match = re.search(r"^- Process max RSS memory:\s*([0-9.]+)\s*MB", summary_text, re.MULTILINE)

    if device_match:
        resource_stats['device'] = device_match.group(1).strip()
    if runtime_match:
        resource_stats['runtime_seconds'] = safe_float(runtime_match.group(1))
    if rss_match:
        resource_stats['max_rss_mb'] = safe_float(rss_match.group(1))

if qualitative_df.empty:
    qualitative_df = pd.DataFrame([
        {
            'filename': 'N/A',
            'top_label': 'N/A',
            'top_score': np.nan,
            'maintenance_priority': 'N/A',
            'analysis_stage': 'N/A',
            'description': 'No qualitative examples available.',
        }
    ])

qualitative_df['top_score'] = qualitative_df.get('top_score', pd.Series(dtype=float)).map(safe_float)

display(Markdown('### Loaded Artifacts Snapshot'))
display(pd.DataFrame([
    {'artifact': 'metrics_json', 'exists': paths['metrics'].exists()},
    {'artifact': 'summary_markdown', 'exists': paths['summary'].exists()},
    {'artifact': 'latency_csv', 'exists': paths['latency_csv'].exists()},
    {'artifact': 'confusion_csv', 'exists': paths['confusion_csv'].exists()},
    {'artifact': 'qualitative_json', 'exists': paths['qualitative'].exists()},
]))

## Dataset Description and Training Details

In [ ]:
normal_count = int(safe_float(dataset.get('normal_count', 0), 0))
abnormal_count = int(safe_float(dataset.get('abnormal_count', 0), 0))
total_count = int(safe_float(dataset.get('total_count', normal_count + abnormal_count), normal_count + abnormal_count))

environment = dataset.get('environment', 'Not specified')
pump_type = dataset.get('pump_type', 'Not specified')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Dataset composition chart
labels = ['Normal clips', 'Abnormal clips']
values = [normal_count, abnormal_count]
colors = ['#4c72b0', '#dd8452']

axes[0].pie(
    values,
    labels=labels,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct * sum(values) / 100.0))})",
    startangle=90,
    colors=colors,
    wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
)
axes[0].set_title('Dataset Composition')
axes[0].text(
    0,
    -1.25,
    f"Total clips: {total_count}\nEnvironment: {environment}\nPump type: {pump_type}",
    ha='center',
    va='top',
    fontsize=9,
)

# Training setup infobox-style panel
axes[1].axis('off')
training_text = (
    "Training Configuration\n"
    f"\nEpochs: {training.get('epochs', 'N/A')}"
    f"\nSplit: {training.get('split', 'N/A')}"
    f"\nMetrics note: {training.get('metrics_note', 'N/A')}"
)
axes[1].text(
    0.02,
    0.95,
    training_text,
    transform=axes[1].transAxes,
    va='top',
    fontsize=10,
    bbox={'boxstyle': 'round,pad=0.6', 'facecolor': '#f5f5f5', 'edgecolor': '#cccccc'},
)
axes[1].set_title('Training Details', pad=10)

plt.tight_layout()
plt.show()


## FFNN Metrics

In [ ]:
main_metrics = {
    'accuracy': safe_float(reliability.get('accuracy')),
    'precision_abnormal': safe_float(reliability.get('precision_abnormal')),
    'recall_abnormal_tpr': safe_float(reliability.get('recall_abnormal_tpr')),
    'specificity_tnr': safe_float(reliability.get('specificity_tnr')),
    'auc': safe_float(reliability.get('auc')),
}
error_metrics = {
    'false_positive_rate': safe_float(reliability.get('false_positive_rate')),
    'false_negative_rate': safe_float(reliability.get('false_negative_rate')),
}

f1_score = np.nan
if not np.isnan(main_metrics['precision_abnormal']) and not np.isnan(main_metrics['recall_abnormal_tpr']):
    p = main_metrics['precision_abnormal']
    r = main_metrics['recall_abnormal_tpr']
    if p + r > 0:
        f1_score = (2 * p * r) / (p + r)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

metric_names = [
    'accuracy',
    'precision_abnormal',
    'recall_abnormal_tpr',
    'specificity_tnr',
    'auc',
]
metric_values = [main_metrics[name] for name in metric_names]
axes[0].bar(metric_names, metric_values, color='#4c72b0')
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Score')
axes[0].set_title('FFNN Reliability Metrics')
axes[0].tick_params(axis='x', rotation=25)
for idx, value in enumerate(metric_values):
    if not np.isnan(value):
        axes[0].text(idx, value + 0.02, f"{value:.3f}", ha='center', fontsize=9)

err_names = list(error_metrics.keys())
err_values = [error_metrics[name] for name in err_names]
axes[1].bar(err_names, err_values, color='#dd8452')
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Rate')
axes[1].set_title('FFNN Error Rates')
axes[1].tick_params(axis='x', rotation=15)
for idx, value in enumerate(err_values):
    if not np.isnan(value):
        axes[1].text(idx, value + 0.02, f"{value:.3f}", ha='center', fontsize=9)

threshold = safe_float(reliability.get('threshold'))
fig.suptitle(
    f"Threshold={threshold:.2f}" + (f" | F1(abnormal)={f1_score:.3f}" if not np.isnan(f1_score) else ''),
    fontsize=11,
)
plt.tight_layout()
plt.show()

# Confusion matrix heatmap recreated from confusion_matrix.csv
cm = confusion_df.apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks(np.arange(confusion_df.shape[1]))
ax.set_yticks(np.arange(confusion_df.shape[0]))
ax.set_xticklabels(confusion_df.columns)
ax.set_yticklabels(confusion_df.index)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix (FFNN Gatekeeper)')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center', color='black', fontsize=11)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Count')
plt.tight_layout()
plt.show()


## Latency and Resource Usage

In [ ]:
if latency_df.empty:
    display(Markdown('> ⚠ No latency data found in `latency_metrics.csv` or `metrics.json`.'))
else:
    latency_plot_df = latency_df.copy()
    latency_plot_df['mode'] = latency_plot_df['mode'].astype(str)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    axes[0].bar(
        latency_plot_df['mode'],
        latency_plot_df['mean_ms'],
        yerr=latency_plot_df['std_ms'].fillna(0.0),
        capsize=4,
        color='#55a868',
    )
    axes[0].set_title('Inference Latency by Mode')
    axes[0].set_ylabel('Latency (ms)')
    axes[0].tick_params(axis='x', rotation=20)

    for i, row in latency_plot_df.iterrows():
        if not np.isnan(row['mean_ms']):
            axes[0].text(i, row['mean_ms'] + 0.8, f"{row['mean_ms']:.2f}", ha='center', fontsize=9)

    resource_names = ['Runtime (s)', 'Max RSS (MB)']
    resource_values = [
        resource_stats['runtime_seconds'],
        resource_stats['max_rss_mb'],
    ]
    bars = axes[1].bar(resource_names, resource_values, color=['#8172b3', '#c44e52'])
    axes[1].set_title(f"Runtime & Memory Context ({resource_stats['device']})")
    axes[1].set_ylabel('Value')

    for bar, value in zip(bars, resource_values):
        if not np.isnan(value):
            axes[1].text(
                bar.get_x() + bar.get_width() / 2,
                value + (value * 0.01 if value > 0 else 0.1),
                f"{value:.2f}",
                ha='center',
                fontsize=9,
            )

    plt.tight_layout()
    plt.show()


## Qualitative Semantic Labels and Maintenance Reports

In [ ]:
priority_colors = {
    'High': '#c44e52',
    'Medium': '#dd8452',
    'Low': '#55a868',
}

qual_plot_df = qualitative_df.copy()
qual_plot_df['maintenance_priority'] = qual_plot_df.get('maintenance_priority', 'N/A').fillna('N/A')
qual_plot_df['top_label'] = qual_plot_df.get('top_label', 'N/A').fillna('N/A')
qual_plot_df['filename'] = qual_plot_df.get('filename', 'N/A').fillna('N/A')
qual_plot_df['top_score'] = qual_plot_df.get('top_score', pd.Series(dtype=float)).map(safe_float)
qual_plot_df = qual_plot_df.sort_values(by='top_score', ascending=True, na_position='first')

fig, ax = plt.subplots(figsize=(12, max(4, 0.7 * len(qual_plot_df))))
bar_colors = [priority_colors.get(p, '#4c72b0') for p in qual_plot_df['maintenance_priority']]
bars = ax.barh(qual_plot_df['filename'], qual_plot_df['top_score'].fillna(0), color=bar_colors)

for bar, (_, row) in zip(bars, qual_plot_df.iterrows()):
    score_value = row['top_score']
    label_text = str(row['top_label'])
    if np.isnan(score_value):
        text = f"{label_text} | score=N/A"
        x_pos = 0.02
    else:
        text = f"{label_text} | score={score_value:.3f}"
        x_pos = score_value + 0.01
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2, text, va='center', fontsize=8.5)

ax.set_xlim(0, 1.02)
ax.set_xlabel('Top semantic-label score')
ax.set_ylabel('Audio clip')
ax.set_title('Qualitative Semantic Labels and Maintenance Priority')

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=color, label=priority)
    for priority, color in priority_colors.items()
]
ax.legend(handles=legend_handles, title='Maintenance priority', loc='lower right')

plt.tight_layout()
plt.show()

display(Markdown('### Qualitative Table (for report text snippets)'))
columns = [
    col
    for col in ['filename', 'top_label', 'top_score', 'maintenance_priority', 'analysis_stage', 'description', 'error']
    if col in qual_plot_df.columns
]
display(qual_plot_df[columns].reset_index(drop=True))


## Appendix: Full Generated Summary

In [ ]:
if summary_text:
    display(Markdown(summary_text))
else:
    show_missing('summary')
